In [1]:
import pandas as pd
df = pd.read_parquet('/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet')
df.tail()

,03LIC_1071.PV,03LIC_1071.OP,02FI_1000.PV,03FIC_1085.OP,03FIC_1085.PV,03FIC_3415.OP,03FIC_3415.PV,03FIC_3435.PV,03FI_1141A.PV,03FI_1151.PV,...,03TIC_1142.OP,03TIC_1142.PV,03TIC_1145.OP,03TIC_1145.PV,03TI_1015.PV,03TI_1081.PV,03TI_1421.PV,03TI_1901.PV,AlarmStatus,AlarmType
TimeStamp,,,,,,,,,,,,,,,,,,,,,
2025-06-23 20:40:00,40.710983,63.101887,8.574106,43.653538,306.86664,10.0,28.186607,99517.820,72216.71,269300.47,...,0.0,-36.081352,0.0,21.851860,17.378136,-32.007050,-0.754356,-31.141922,OFF,
2025-06-23 20:41:00,41.379030,62.556927,8.546025,43.834140,310.23320,10.0,28.044992,99575.984,72216.71,274164.06,...,0.0,-36.081352,0.0,22.105950,17.643166,-31.893760,-0.527779,-31.065750,OFF,
2025-06-23 20:42:00,47.099087,60.847668,8.579161,43.861470,311.68207,10.0,27.242529,99098.984,72216.71,272590.56,...,0.0,-36.123700,0.0,22.021255,17.477524,-31.879599,-0.513618,-31.065750,OFF,
2025-06-23 20:43:00,38.902195,63.382630,8.504224,44.953030,302.43472,10.0,27.366440,98749.980,72216.71,271760.88,...,0.0,-36.039005,0.0,21.894203,17.361572,-31.922081,-0.471142,-31.065750,OFF,
2025-06-23 20:44:00,40.734318,63.342285,8.542656,45.167770,304.48022,10.0,27.502150,99064.090,72216.71,276367.00,...,0.0,-36.081352,0.0,22.091835,17.643166,-31.808798,-0.499455,-31.008621,OFF,


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('/home/h604827/ControlActions')

# ── 1. Load the 1071 workbook ─────────────────────────────────────────────
workbook_path = BASE_DIR / 'DATA/1071_pvlo_alarms_clustered_with_control_actions_with_plant_context.xlsx'

print("Loading 1071 workbook...")
alarm_clusters_df = pd.read_excel(workbook_path, sheet_name='alarm_clusters')
control_actions_df = pd.read_excel(workbook_path, sheet_name='control_actions')

alarm_clusters_df['cluster_start_time'] = pd.to_datetime(alarm_clusters_df['cluster_start_time'])
alarm_clusters_df['cluster_end_time'] = pd.to_datetime(alarm_clusters_df['cluster_end_time'])

# Get unique clusters
clusters = alarm_clusters_df.groupby('cluster_id').agg(
    cluster_start=('cluster_start_time', 'first'),
    cluster_end=('cluster_end_time', 'first')
).reset_index()

print(f"  Alarm clusters: {len(clusters)} unique clusters, {len(alarm_clusters_df)} alarm rows")
print(f"  Control actions: {len(control_actions_df)} rows")

# ── 2. Load RCA CSV file ──────────────────────────────────────────────────
rca_csv_path = BASE_DIR / 'DATA/RCA_Validated_results_FI1000_Final_with_paths 5(in).csv'
rca_df = pd.read_csv(rca_csv_path)
rca_df['AlarmStart_rounded'] = pd.to_datetime(rca_df['AlarmStart_rounded'])
# Drop rows with NaN AlarmStart_rounded (last 2 rows are empty)
rca_df = rca_df.dropna(subset=['AlarmStart_rounded']).reset_index(drop=True)
print(f"  RCA CSV: {len(rca_df)} valid alarms")

# ── 3. Load Bucket info from Updated Ground Truth ─────────────────────────
bucket_path = BASE_DIR / 'DATA/Updated Ground truth -Adnoc RCA_bucket 2.xlsx'
bucket_df = pd.read_excel(bucket_path)
bucket_per_alarm = bucket_df.groupby('AlarmStart_rounded')['Bucket'].first().reset_index()
bucket_per_alarm['AlarmStart_rounded'] = pd.to_datetime(bucket_per_alarm['AlarmStart_rounded'])
print(f"  Bucket info: {len(bucket_per_alarm)} alarms with bucket values")

# ── 4. Match each RCA alarm to nearest cluster ────────────────────────────
cluster_starts = clusters['cluster_start'].values

matches = []
for i, row in rca_df.iterrows():
    rca_start = row['AlarmStart_rounded']
    diffs = np.abs(cluster_starts - np.datetime64(rca_start))
    nearest_idx = np.argmin(diffs)
    nearest_cluster = clusters.iloc[nearest_idx]
    
    matches.append({
        'rca_idx': i,
        'rca_alarm_start': rca_start,
        'matched_cluster_id': int(nearest_cluster['cluster_id']),
    })

matches_df = pd.DataFrame(matches)
print(f"\n  Matched all {len(rca_df)} RCA alarms to clusters.")

# ── 5. Extract control actions for the matched clusters ───────────────────
matched_cluster_ids = matches_df['matched_cluster_id'].unique()
print(f"  Unique cluster IDs to extract: {len(matched_cluster_ids)}")

gt_control_actions = control_actions_df[
    control_actions_df['cluster_id'].isin(matched_cluster_ids)
].copy().reset_index(drop=True)

print(f"  Control actions for these clusters: {len(gt_control_actions)} rows")

# ── 6. Map cluster_id → rca_idx and select columns ───────────────────────
cluster_to_rca = matches_df.set_index('matched_cluster_id')['rca_idx'].to_dict()
gt_control_actions['rca_idx'] = gt_control_actions['cluster_id'].map(cluster_to_rca)

keep_cols = ['rca_idx', 'action_timing', 'action_direction', 'Source', 
             'Description', 'VT_Start', 'PrevValue', 'Value', 'Step']
gt_control_actions = gt_control_actions[keep_cols].copy()

# ── 7. Compute most operated tags per alarm episode ───────────────────────
# Count operations per (rca_idx, Source) and order by count descending
ops_per_tag = gt_control_actions.groupby(['rca_idx', 'Source']).size().reset_index(name='count')
ops_per_tag = ops_per_tag.sort_values(['rca_idx', 'count'], ascending=[True, False])

# Build ordered tag list string per rca_idx
most_operated = ops_per_tag.groupby('rca_idx').apply(
    lambda g: ', '.join(f"{row['Source']}({row['count']})" for _, row in g.iterrows())
).reset_index(name='most_operated_tags')

print(f"\n  Most operated tags computed for {len(most_operated)} episodes")

# ── 8. Build first sheet: RCA CSV + Bucket + most_operated_tags ───────────
mapping_sheet = rca_df.copy()
mapping_sheet.insert(0, 'rca_idx', range(len(mapping_sheet)))

# Merge Bucket
mapping_sheet = mapping_sheet.merge(
    bucket_per_alarm[['AlarmStart_rounded', 'Bucket']], 
    on='AlarmStart_rounded', how='left'
)

# Merge most_operated_tags
mapping_sheet = mapping_sheet.merge(most_operated, on='rca_idx', how='left')

print(f"  First sheet columns: {list(mapping_sheet.columns)}")

# ── 9. Save to Excel ──────────────────────────────────────────────────────
output_path = BASE_DIR / 'RESULTS/ground_truth_81_episodes_control_actions.xlsx'
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    mapping_sheet.to_excel(writer, sheet_name='alarm_mapping', index=False)
    gt_control_actions.to_excel(writer, sheet_name='control_actions', index=False)

print(f"\n✓ Saved to: {output_path}")
print(f"  Sheet 'alarm_mapping': {len(mapping_sheet)} rows, {len(mapping_sheet.columns)} columns")
print(f"  Sheet 'control_actions': {len(gt_control_actions)} rows")
print(f"\nBucket coverage: {mapping_sheet['Bucket'].notna().sum()} / {len(mapping_sheet)} alarms have bucket values")
print(f"Most operated tags coverage: {mapping_sheet['most_operated_tags'].notna().sum()} / {len(mapping_sheet)} alarms")
print(f"\nSample most_operated_tags:")
print(mapping_sheet[['rca_idx', 'AlarmStart_rounded', 'Bucket', 'most_operated_tags']].head(10).to_string())

Loading 1071 workbook...
  Alarm clusters: 539 unique clusters, 1379 alarm rows
  Control actions: 16094 rows
  RCA CSV: 80 valid alarms
  Bucket info: 81 alarms with bucket values

  Matched all 80 RCA alarms to clusters.
  Unique cluster IDs to extract: 74
  Control actions for these clusters: 5215 rows


/tmp/ipykernel_241277/983608418.py:84: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  most_operated = ops_per_tag.groupby('rca_idx').apply(



  Most operated tags computed for 71 episodes
  First sheet columns: ['rca_idx', 'AlarmStart_rounded', 'predicted_tags', 'SIT Validation results ', 'SIT Validation results .1', 'Accuracy', 'False positive', 'gt_cause1', 'gt_cause2', 'gt_cause3', 'any_exact_hit', 'any_pipe_equip_hit', 'any_pipeline_only_hit', 'any_equipment_only_hit', 'Eliminated Effect Tags', 'Bucket', 'most_operated_tags']

✓ Saved to: /home/h604827/ControlActions/RESULTS/ground_truth_81_episodes_control_actions.xlsx
  Sheet 'alarm_mapping': 80 rows, 17 columns
  Sheet 'control_actions': 5215 rows

Bucket coverage: 80 / 80 alarms have bucket values
Most operated tags coverage: 71 / 80 alarms

Sample most_operated_tags:
   rca_idx  AlarmStart_rounded Bucket                                                                                                                                                                                                                                              most_operated_tags
0        